# Chapter 13. Applying PPO to Continuous Control Environments — Practice Notebook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SuminHan/book-ml/blob/main/notebooks/ml2/chapter13_3_ppo_reacher.ipynb)

Book body text: [13.3 Applying PPO to Continuous Control Environments](https://smhanlab.com/book-ml/kor/ml2/chapter13/3.html)

This notebook applies **the very same PPO code you have been using** (with no changes)
to the 2-joint robot arm task `Reacher-v5` and confirms three things:

1. **Gaussian policy** (mean/std output) is all that's needed to parameterize continuous actions —
   the log-probability is just the log density of a normal distribution.
2. **The same PPO code learns Reacher as-is** — only the environment name changes,
   and it surpasses a hand-crafted proportional controller.
3. **The two-curve phenomenon** — the noisy episodes gathered *during* learning can
   look worse, while the deterministic (mean-output) policy that is actually deployed
   on a robot is exactly the one that improves the most.

Only numpy/matplotlib/torch(cpu) are used — no external data download is needed
(`Reacher-v5` ships with `gymnasium[mujoco]`).

In [1]:
import math, time
import numpy as np
import matplotlib
matplotlib.use("Agg")
matplotlib.rcParams["font.sans-serif"] = ["Noto Sans CJK KR", "NanumGothic", "DejaVu Sans"]
matplotlib.rcParams["axes.unicode_minus"] = False
import matplotlib.pyplot as plt
import torch, torch.nn as nn
from torch.distributions import Normal
import gymnasium as gym

import os
IMG = "/home/smhan/book-ml/kor/src/images"
if not os.path.isdir(IMG):   # auto-fallback to /tmp on Colab etc.
    IMG = "/tmp"
print("torch", torch.__version__, "| gymnasium", gym.__version__)
print(f"Figure save location: {IMG}")
print("Reproducible seed: all randomness in this notebook is seeded with 42")

torch 2.13.0+cpu | gymnasium 1.3.0
Figure save location: /home/smhan/book-ml/kor/src/images
Reproducible seed: all randomness in this notebook is seeded with 42


## 1. Reacher-v5: The 2-joint robot arm's "touch the target" task

`Reacher-v5` is a 2-joint (shoulder + elbow) planar robot arm whose task is to
**move the arm's end (hand/finger) to a randomly generated target point**
within a 100-step horizon.
The state is a 10-dimensional vector (target position 2 + hand position 2 + elbow position 2 +
shoulder-joint position 2 + hand velocity 2) and the action is 2-dimensional continuous torque
(range [-1, +1]).

The per-step reward is closer to 0 the nearer the hand is to the target
(`-1.0 * distance - 0.01 * ||action||^2`).
That is, **Reacher's return is always negative and closer to 0 is better**
(same sign convention as Pendulum in 11.2).

In [2]:
env = gym.make("Reacher-v5")
obs, info = env.reset(seed=0)
print("State space: ", env.observation_space.shape)
print("Action space: ", env.action_space.shape, "range:", env.action_space.low, "~", env.action_space.high)
print("One state example (float64):", np.round(obs, 3))
print()
print("State breakdown (10-dim):")
for lo, hi, name in [(0, 2, "target position"), (2, 4, "hand (finger) position"), (4, 6, "elbow position"),
                     (6, 8, "shoulder-joint position"), (8, 10, "hand velocity")]:
    print(f"  obs[{lo}:{hi}] = {np.round(obs[lo:hi], 3)}   <- {name}")
print()
print("Reward = -1.0 * (hand-target distance) - 0.01 * ||action||^2   (per step)")

# baseline: return of a random policy (10 episodes)
rets = []
for ep in range(10):
    s, _ = env.reset(seed=42 + ep)
    tot = 0.0
    for _ in range(100):
        s, r, term, trunc, _ = env.step(env.action_space.sample())
        tot += r
        if term or trunc:
            break
    rets.append(tot)
rand_mean = float(np.mean(rets))
print()
print(f"Random policy 10-episode mean return: {rand_mean:.2f}   (per episode: {[round(x,1) for x in rets]})")
env.close()

State space:  (10,)
Action space:  (2,) range: [-1. -1.] ~ [1. 1.]
One state example (float64): [ 1.     0.999  0.027 -0.046  0.043  0.092  0.     0.004  0.167 -0.091]

State breakdown (10-dim):
  obs[0:2] = [1.    0.999]   <- target position
  obs[2:4] = [ 0.027 -0.046]   <- hand (finger) position
  obs[4:6] = [0.043 0.092]   <- elbow position
  obs[6:8] = [0.    0.004]   <- shoulder-joint position
  obs[8:10] = [ 0.167 -0.091]   <- hand velocity

Reward = -1.0 * (hand-target distance) - 0.01 * ||action||^2   (per step)

Random policy 10-episode mean return: -44.37   (per episode: [np.float64(-44.9), np.float64(-38.3), np.float64(-51.8), np.float64(-47.3), np.float64(-46.5), np.float64(-48.2), np.float64(-35.0), np.float64(-42.6), np.float64(-44.8), np.float64(-44.2)])


## 2. Baseline: a hand-crafted proportional controller

Before RL, the "hand-crafted rule" baseline from 13.2:
**`action = -k * (hand position - target position)`** — the further the hand is from the
target, the stronger the torque in the opposite direction. This is not reinforcement
learning, it is a one-line classical control rule (a simplified PD controller).
With `k=0.5` we measure its return and use it as the benchmark for "can RL beat a simple rule?".

In [3]:
def prop_controller(k=0.5, seed=100, n_ep=10):
    env = gym.make("Reacher-v5")
    rets = []
    for ep in range(n_ep):
        s, _ = env.reset(seed=seed + ep)
        tot = 0.0
        for _ in range(100):
            a = np.clip(-k * (s[2:4] - s[0:2]), -1.0, 1.0)   # proportional control: pull hand to target
            s, r, term, trunc, _ = env.step(a)
            tot += r
            if term or trunc:
                break
        rets.append(tot)
    env.close()
    return rets

prop = prop_controller()
prop_mean = float(np.mean(prop))
print(f"Proportional controller (k=0.5) 10-episode mean return: {prop_mean:.2f}")
print(f"   (per episode: {[round(x,1) for x in prop]})")

Proportional controller (k=0.5) 10-episode mean return: -25.95
   (per episode: [np.float64(-22.7), np.float64(-21.8), np.float64(-25.1), np.float64(-24.6), np.float64(-20.3), np.float64(-27.1), np.float64(-29.4), np.float64(-30.3), np.float64(-30.7), np.float64(-27.3)])


## 3. Gaussian policy: log-probability = log of the normal density

In a continuous action space the policy outputs the **mean and standard deviation** of a
normal distribution for each action dimension, and "the probability of action a" is the
*density* of that normal distribution:

$$\pi_\theta(a|s) = \mathcal{N}(a;\, \mu_\theta(s),\, \sigma_\theta(s)^2)$$

Torch's `Normal.log_prob` computes exactly
`-0.5*log(2*pi*sigma^2) - (a-mu)^2/(2*sigma^2)`.
Note: because a **density** can exceed 1 (when sigma is small),
the log-probability can even be **positive** — it is a log of a *density*, not a log of a
probability in [0,1].

In [4]:
def glp(a, mu, s):          # single action dimension, = the textbook formula
    return -0.5 * math.log(2 * math.pi * s**2) - (a - mu)**2 / (2 * s**2)

print("Single action dimension (scalar a) — matches the textbook formula exactly:")
print(f"  log N(0; 0, 1^2)   = {glp(0.0, 0.0, 1.0):.4f}   (at the mean, standard normal)")
print(f"  log N(2; 0, 1^2)   = {glp(2.0, 0.0, 1.0):.4f}   (2 sigma away -> much smaller / more negative)")
print(f"  torch check:       {Normal(torch.tensor([0.0]), torch.tensor([1.0])).log_prob(torch.tensor([2.0])).item():.4f}  (identical)")

# a density can exceed 1 once sigma is small enough -> the log-probability goes POSITIVE
print()
print(f"log N(1; 1, 0.25^2) = {glp(1.0, 1.0, 0.25):.4f}   (POSITIVE! density at the mean = 1/(0.25*sqrt(2*pi)) ≈ 1.60 > 1)")
print("So 'log-probability' is a log of a DENSITY, not a log of a probability in [0,1] — it may be > 0.")
print()
print("(In the PPO code below, the two action dimensions are summed: full log-prob = sum over dims.)")
a_s = Normal(torch.tensor([0.0, 0.0]), torch.tensor([1.0, 1.0])).sample()
print(f"sample from a 2-dim Gaussian policy: {a_s.tolist()}   (differs each call)")

Single action dimension (scalar a) — matches the textbook formula exactly:
  log N(0; 0, 1^2)   = -0.9189   (at the mean, standard normal)
  log N(2; 0, 1^2)   = -2.9189   (2 sigma away -> much smaller / more negative)
  torch check:       -2.9189  (identical)

log N(1; 1, 0.25^2) = 0.4674   (POSITIVE! density at the mean = 1/(0.25*sqrt(2*pi)) ≈ 1.60 > 1)
So 'log-probability' is a log of a DENSITY, not a log of a probability in [0,1] — it may be > 0.

(In the PPO code below, the two action dimensions are summed: full log-prob = sum over dims.)
sample from a 2-dim Gaussian policy: [0.33400726318359375, -1.0575684309005737]   (differs each call)


## 4. PPO: learning on Reacher (150 iterations x 512 steps = 76,800 steps)

The network and the update loop are the same as 11.2 — `GaussianPolicy` (shared trunk
+ mu head + learnable `log_std`), `ValueNet` (Critic), GAE, clipped loss + value loss -
entropy bonus. The only Reacher-specific differences are two: input 10-dim / output 2-dim,
and **clamping `log_std <= log(1) = 0`** (i.e. capping sigma at 1.0) — because Reacher's
action range is [-1, 1], letting sigma grow unbounded would let exploration noise swamp
the policy (a common mistake examined in the main text).

At the same time we record **two learning curves**: the return of the *stochastic*
evaluation episodes run *during* learning (with sampling noise), and the return of the
*deterministic* policy that outputs only the mean mu (the deployable form on a robot).

In [5]:
class GaussianPolicy(nn.Module):
    def __init__(self, state_dim, act_dim, act_high=1.0):
        super().__init__()
        self.register_buffer("act_high", torch.full((act_dim,), act_high))
        self.net = nn.Sequential(nn.Linear(state_dim, 64), nn.Tanh(), nn.Linear(64, 64), nn.Tanh())
        self.mu = nn.Linear(64, act_dim)
        self.log_std = nn.Parameter(torch.zeros(act_dim) - 0.5)   # initial sigma = e^-0.5 ≈ 0.61
    def forward(self, x):
        h = self.net(x)
        mu = torch.tanh(self.mu(h)) * self.act_high     # scale the mean to the action range
        std = torch.exp(self.log_std)
        return mu, std

class ValueNet(nn.Module):
    def __init__(self, state_dim):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(state_dim, 64), nn.Tanh(), nn.Linear(64, 64), nn.Tanh(), nn.Linear(64, 1))
    def forward(self, x):
        return self.net(x).squeeze(-1)

def gae(rewards, values, dones, gamma, lam):
    adv, g = np.zeros_like(rewards), 0.0
    for t in reversed(range(len(rewards))):
        last = 1.0 if t == len(rewards) - 1 else 0.0
        delta = rewards[t] + gamma * values[t + 1] * (1 - last) - values[t]
        g = delta + gamma * lam * (1 - last) * g
        adv[t] = g
    return adv, adv + values[:-1]

def evaluate(net, deterministic, seed=1000, n_ep=10):
    env = gym.make("Reacher-v5"); rets = []
    for ep in range(n_ep):
        s, _ = env.reset(seed=seed + ep); tot = 0.0
        for _ in range(100):
            st = torch.tensor(np.asarray(s, dtype=np.float32))
            with torch.no_grad():
                mu, std = net(st)
                a = mu.numpy() if deterministic else Normal(mu, std).sample().numpy()
            s, r, t_, tr_, _ = env.step(np.clip(a, -1, 1))
            tot += r
            if t_ or tr_:
                break
        rets.append(tot)
    env.close()
    return float(np.mean(rets))

torch.manual_seed(42); np.random.seed(42)
env = gym.make("Reacher-v5")
actor, critic = GaussianPolicy(10, 2, 1.0), ValueNet(10)
opt = torch.optim.Adam(list(actor.parameters()) + list(critic.parameters()), lr=3e-4)
EPS, GAMMA, LAM, N_EPOCH, MB = 0.2, 0.99, 0.95, 4, 64
N_ITER, STEPS = 150, 512

curves_sampled, curves_det, logstd_hist = [], [], []
t0 = time.time()
for it in range(N_ITER):
    s, _ = env.reset()
    buf = []
    for _ in range(STEPS):
        s_old = np.asarray(s, dtype=np.float32)
        st = torch.tensor(s_old)
        with torch.no_grad():
            mu, std = actor(st)
            d = Normal(mu, std)
            a_t = d.sample()
            lp = d.log_prob(a_t).sum().item()
            v = critic(st).item()
        s, r, term, trunc, _ = env.step(np.clip(a_t.numpy(), -1.0, 1.0))
        buf.append((s_old, a_t.numpy(), r, float(term or trunc), lp, v))
        if term or trunc:
            s, _ = env.reset()
    states = np.stack([b[0] for b in buf])
    acts   = torch.tensor(np.stack([b[1] for b in buf]))
    rews   = np.array([b[2] for b in buf]); dones = np.array([b[3] for b in buf])
    lpo    = torch.tensor(np.array([b[4] for b in buf]))
    vs     = np.append(np.array([b[5] for b in buf]), 0.0)
    adv, rets = gae(rews, vs, dones, GAMMA, LAM)
    adv = (adv - adv.mean()) / (adv.std() + 1e-8)          # advantage normalization (11.3)
    Rb = torch.tensor(rets, dtype=torch.float32); Ab = torch.tensor(adv, dtype=torch.float32)
    Sb = torch.tensor(states, dtype=torch.float32)
    perm = np.random.permutation(len(buf))
    for _ in range(N_EPOCH):                               # reuse the same data 4 epochs
        for start in range(0, len(buf), MB):
            sel = perm[start:start + MB]
            mu, std = actor(Sb[sel]); d = Normal(mu, std)
            ratio = torch.exp(d.log_prob(acts[sel]).sum(dim=1) - lpo[sel])
            surr = torch.min(ratio * Ab[sel], torch.clamp(ratio, 1 - EPS, 1 + EPS) * Ab[sel])
            v = critic(Sb[sel])
            loss = -surr.mean() + 0.5 * nn.functional.mse_loss(v, Rb[sel]) - 0.01 * d.entropy().sum()
            opt.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(list(actor.parameters()) + list(critic.parameters()), 0.5)
            opt.step()
            with torch.no_grad():
                actor.log_std.clamp_(max=math.log(1.0))     # cap sigma at 1.0 (action range [-1, 1])

    # two curves: stochastic (with sampling noise) vs deterministic (mu only)
    curves_sampled.append(evaluate(actor, deterministic=False, seed=1000 + it * 10))
    curves_det.append(evaluate(actor, deterministic=True, seed=1000 + it * 10))
    logstd_hist.append(actor.log_std.detach().clone().tolist())
    if it % 25 == 0 or it == N_ITER - 1:
        print(f"iteration {it:3d}  stochastic last10={curves_sampled[-1]:8.2f}   "
              f"deterministic last10={curves_det[-1]:8.2f}   elapsed={time.time()-t0:5.1f}s")
env.close()
print(f"\nTraining complete: {N_ITER*STEPS:,} steps in {time.time()-t0:.1f} s (CPU)")

iteration   0  stochastic last10=  -41.67   deterministic last10=  -10.71   elapsed=  0.3s
iteration  25  stochastic last10=  -48.96   deterministic last10=  -11.47   elapsed=  6.4s
iteration  50  stochastic last10=  -56.51   deterministic last10=  -10.47   elapsed= 12.6s
iteration  75  stochastic last10=  -63.58   deterministic last10=   -9.98   elapsed= 18.8s
iteration 100  stochastic last10=  -61.21   deterministic last10=  -11.27   elapsed= 24.8s
iteration 125  stochastic last10=  -63.34   deterministic last10=  -12.45   elapsed= 30.8s
iteration 149  stochastic last10=  -66.52   deterministic last10=  -31.02   elapsed= 36.6s

Training complete: 76,800 steps in 36.6 s (CPU)


## 5. Learning curves: the two-curve phenomenon

The two curves tell different stories. The **deterministic** (deployable) policy improves
steadily from ≈ the random baseline to well below the hand-crafted controller. The
**stochastic** (during-learning) curve, by contrast, first improves and then degrades —
the growing exploration noise (sigma) makes each *sampled* episode's return worse even as
the policy's mean (the actual controller) gets better.

In [6]:
x = np.arange(1, len(curves_det) + 1)
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(x, curves_det, color="tab:blue", linewidth=2.2,
        label="Deterministic policy (mu only) — deployable on robot")
ax.plot(x, curves_sampled, color="tab:orange", linewidth=1.4, alpha=0.85,
        label="Stochastic episodes (sampling noise included)")
ax.axhline(prop_mean, color="gray", linestyle="--", linewidth=1.2,
           label=f"Hand-crafted proportional controller ≈ {prop_mean:.1f}")
ax.axhline(rand_mean, color="silver", linestyle=":", linewidth=1.4,
           label=f"Random policy ≈ {rand_mean:.1f}")
ax.set_xlabel("Iteration (512 steps each)")
ax.set_ylabel("Mean return over 10 episodes (closer to 0 is better)")
ax.set_title("PPO on Reacher-v5 — two learning curves (seed 42, 76,800 steps)")
ax.invert_yaxis()   # return is negative; closer to 0 (top) is better
ax.legend(fontsize=8, loc="lower right")
plt.tight_layout()
p = os.path.join(IMG, "ch13_3_ppo_reacher_curves.svg")
plt.savefig(p, bbox_inches="tight")
plt.show()
print(f"SVG saved: {p}")
print(f"\nDeterministic final: {curves_det[-1]:.2f}   (random baseline {rand_mean:.2f}, hand-crafted {prop_mean:.2f})")
print(f"Stochastic    final: {curves_sampled[-1]:.2f}")
print(f"Final sigma (two action dims): {[round(math.exp(v), 2) for v in logstd_hist[-1]]}")

SVG saved: /home/smhan/book-ml/kor/src/images/ch13_3_ppo_reacher_curves.svg

Deterministic final: -31.02   (random baseline -44.37, hand-crafted -25.95)
Stochastic    final: -66.52
Final sigma (two action dims): [1.0, 1.0]


## 6. Summary

| What we confirmed | Conclusion |
|---|---|
| Gaussian log-probability | `-0.5*log(2*pi*sigma^2) - (a-mu)^2/(2*sigma^2)`; can be **positive** when the density exceeds 1 |
| Random vs hand-crafted vs PPO (deterministic) | -44.4 / -25.9 / -31.0 — PPO beats a one-line hand-crafted rule |
| Two-curve phenomenon | The *noisy* learning episodes can degrade while the *deployable* mean policy improves — evaluate a robot with its mean policy, not with exploration noise |

> "Parameterize the action as a Gaussian, and the whole PPO machine (clipping, GAE,
> entropy bonus) keeps working as-is" — that is the generality of policy-based methods,
> and why the same code that trained Pendulum in 11.2 can now train a 2-joint arm.
> The deployable artifact is the **mean** mu_theta(s), and its quality is what
> the deterministic curve tracks.